# PoseCoach P02 — Fit3D: Steps 11b → 12  (Google Colab · Clean Edition)

> **Data:** Google Drive `GYMVISION AI/datasets/fit3d/`  
> Run **Runtime → Run all** on a fresh session — every section is self-contained.

| Step | What it does | Output |
|------|-------------|--------|
| **11b** | Explore Fit3D dataset structure | Console summary |
| **11c** | Compute Golden Angle Templates from 3D skeletons | `golden_angle_ranges.json` |
| **12**  | Validate rep counter (v5) with Fit3D ground truth | `rep_counter_validation.json` |

```
MyDrive/GYMVISION AI/datasets/fit3d/
├── fit3d_info.json
└── raw/
    ├── train/   s03 s04 s05 s07 s08 s09 s10 s11  (8 subjects)
    └── test/    s02 s12 s13                       (3 subjects)
        each subject/
            ├── joints3d_25/  <exercise>.json  ← [N_frames, 25, 3]
            └── rep_ann.json                   ← list of peak frame indices per exercise
```

---
## 🔧 Setup — Mount Drive & Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time
import numpy as np
from pathlib import Path
from collections import defaultdict
from scipy.signal import find_peaks
from scipy.ndimage import gaussian_filter1d
from numpy.fft import fft, ifft

# ── Paths ─────────────────────────────────────────────────────────────────────
DRIVE_ROOT          = '/content/drive/MyDrive'
GYMVISION_ROOT      = f'{DRIVE_ROOT}/GYMVISION AI'
FIT3D_BASE          = f'{GYMVISION_ROOT}/datasets/fit3d'
FIT3D_RAW           = f'{FIT3D_BASE}/raw'
FIT3D_INFO_PATH     = f'{FIT3D_BASE}/fit3d_info.json'
ANGLE_TEMPLATES_DIR = f'{FIT3D_BASE}/angle_templates'
EVAL_DIR            = f'{GYMVISION_ROOT}/data/eval'

for d in [ANGLE_TEMPLATES_DIR, EVAL_DIR]:
    os.makedirs(d, exist_ok=True)

train_subjects = sorted(os.listdir(f'{FIT3D_RAW}/train'))
test_subjects  = sorted(os.listdir(f'{FIT3D_RAW}/test'))
print(f'✅ Drive mounted')
print(f'   Train subjects ({len(train_subjects)}): {train_subjects}')
print(f'   Test  subjects ({len(test_subjects)}):  {test_subjects}')

---
## ✅ Step 11b — Explore Fit3D Dataset Structure

Maps out subjects, exercises, and file counts.  
Also confirms the JSON format: `{joints3d_25: [N_frames, 25, 3]}`.

In [ ]:
import json, os
from pathlib import Path
from collections import defaultdict
import numpy as np

fit3d_root = Path(FIT3D_RAW)

# ── 1. Dataset info ───────────────────────────────────────────────────────────
with open(FIT3D_INFO_PATH) as f:
    fit3d_info = json.load(f)
print('fit3d_info.json keys:', list(fit3d_info.keys()))

# ── 2. Walk structure ─────────────────────────────────────────────────────────
exercise_set = set()
print()
for split in ['train', 'test']:
    split_path = fit3d_root / split
    if not split_path.exists():
        print(f'  ⚠️  {split}/ not found'); continue
    subjects = sorted(os.listdir(split_path))
    print(f'[{split}] {len(subjects)} subjects')
    for subj in subjects:
        j3d_path = split_path / subj / 'joints3d_25'
        if j3d_path.exists():
            exs = sorted(j3d_path.glob('*.json'))
            exercise_set.update(f.stem for f in exs)
            print(f'  {subj}  joints3d_25/  → {len(exs)} exercises', end='')
        else:
            print(f'  {subj}  ⚠️  no joints3d_25/', end='')
        rep_f = split_path / subj / 'rep_ann.json'
        if rep_f.exists():
            with open(rep_f) as f: ra = json.load(f)
            print(f'  |  rep_ann.json → {len(ra)} sequences')
        else:
            print()

print(f'\nTotal unique exercises: {len(exercise_set)}')

# ── 3. Inspect one JSON ───────────────────────────────────────────────────────
sample_subj = sorted(os.listdir(f'{FIT3D_RAW}/train'))[0]
sample_file = sorted((Path(FIT3D_RAW)/'train'/sample_subj/'joints3d_25').glob('*.json'))[0]
with open(sample_file) as f: sample_data = json.load(f)

JOINTS_KEY = None
for key, val in sample_data.items():
    if isinstance(val, list):
        arr = np.array(val)
        if arr.ndim == 3 and arr.shape[-1] == 3:
            JOINTS_KEY = key; break

print(f'\nSample: {sample_file.name}')
print(f'  Key: "{JOINTS_KEY}"  shape: {np.array(sample_data[JOINTS_KEY]).shape}')
print(f'  Range: [{np.array(sample_data[JOINTS_KEY]).min():.3f}, {np.array(sample_data[JOINTS_KEY]).max():.3f}] metres')
JOINTS_KEY = JOINTS_KEY or 'joints3d_25'

---
## ✅ Step 11c — Compute Golden Angle Templates from 3D Skeletons

### Fit3D 25-Joint Skeleton — limb-first ordering (verified by z-height diagnostics)
```
 0: pelvis       1: l_hip        2: l_knee       3: l_ankle
 4: r_hip        5: r_knee       6: r_ankle
 7: spine        8: neck         9: head        10: head_top
11: l_shoulder  12: l_elbow     13: l_wrist
14: r_shoulder  15: r_elbow     16: r_wrist
17: l_foot      18: l_foot_2    19: r_foot      20: r_foot_2
21: l_hand      22: l_hand_2    23: r_hand      24: r_hand_2
```
> Verified: joint[1]→[2] = 0.471 m (thigh), joint[2]→[3] = 0.461 m (shank)

In [ ]:
import numpy as np

# ── Joint names — limb-first ordering ────────────────────────────────────────
JOINT_NAMES = [
    'pelvis',     # 0
    'l_hip',      # 1
    'l_knee',     # 2
    'l_ankle',    # 3
    'r_hip',      # 4
    'r_knee',     # 5
    'r_ankle',    # 6
    'spine',      # 7
    'neck',       # 8
    'head',       # 9
    'head_top',   # 10
    'l_shoulder', # 11
    'l_elbow',    # 12
    'l_wrist',    # 13
    'r_shoulder', # 14
    'r_elbow',    # 15
    'r_wrist',    # 16
    'l_foot',     # 17
    'l_foot_2',   # 18
    'r_foot',     # 19
    'r_foot_2',   # 20
    'l_hand',     # 21
    'l_hand_2',   # 22
    'r_hand',     # 23
    'r_hand_2',   # 24
]

# ── Angle triplets: (joint_A, vertex_B, joint_C) ─────────────────────────────
ANGLE_TRIPLETS = {
    'left_knee_angle':     (1,  2,  3),   # l_hip  → l_knee  ← l_ankle
    'right_knee_angle':    (4,  5,  6),   # r_hip  → r_knee  ← r_ankle
    'left_hip_angle':      (7,  1,  2),   # spine  → l_hip   ← l_knee
    'right_hip_angle':     (7,  4,  5),   # spine  → r_hip   ← r_knee
    'left_ankle_angle':    (2,  3, 17),   # l_knee → l_ankle ← l_foot
    'right_ankle_angle':   (5,  6, 19),   # r_knee → r_ankle ← r_foot
    'left_elbow_angle':    (11, 12, 13),  # l_shoulder → l_elbow ← l_wrist
    'right_elbow_angle':   (14, 15, 16),  # r_shoulder → r_elbow ← r_wrist
    'left_shoulder_angle': (7,  11, 12),  # spine → l_shoulder ← l_elbow
    'right_shoulder_angle':(7,  14, 15),  # spine → r_shoulder ← r_elbow
    'trunk_flex':          (8,   7,  0),  # neck  → spine    ← pelvis
    'hip_trunk_angle':     (1,   0,  4),  # l_hip → pelvis   ← r_hip
}

print('✅ Joint mapping loaded (limb-first ordering)')
for name, (a, b, c) in ANGLE_TRIPLETS.items():
    print(f'  {name:30s}  {JOINT_NAMES[a]:12s} → {JOINT_NAMES[b]:12s} ← {JOINT_NAMES[c]}')

In [ ]:
import json
import numpy as np

# ── Core helpers ──────────────────────────────────────────────────────────────
def compute_angle_3d(a, b, c):
    """Angle (degrees) at joint b, formed by vectors b→a and b→c."""
    ba = a - b;  bc = c - b
    cos_a = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-8)
    return float(np.degrees(np.arccos(np.clip(cos_a, -1.0, 1.0))))

def compute_angles_sequence(joints_seq, triplets):
    """Return {angle_name: np.array([N_frames])} for a full sequence."""
    result = {name: [] for name in triplets}
    for frame in joints_seq:
        for name, (ia, ib, ic) in triplets.items():
            result[name].append(compute_angle_3d(frame[ia], frame[ib], frame[ic]))
    return {k: np.array(v) for k, v in result.items()}

def load_joints3d(json_path):
    """Load [N_frames, 25, 3] array from a Fit3D JSON file."""
    with open(json_path) as f:
        data = json.load(f)
    for key in [JOINTS_KEY, 'joints3d_25', 'joints3d', 'joints']:
        if key in data:
            arr = np.array(data[key])
            if arr.ndim == 3 and arr.shape[-1] == 3:
                return arr
    for val in data.values():
        if isinstance(val, list):
            arr = np.array(val)
            if arr.ndim == 3 and arr.shape[-1] == 3:
                return arr
    return None

print('✅ Helper functions defined')

In [ ]:
# ── Compute angle distributions across ALL train subjects ─────────────────────
# Takes ~5–15 min depending on Drive speed.

import time
from pathlib import Path
from collections import defaultdict

angle_stats = defaultdict(lambda: defaultdict(list))  # {exercise: {angle: [values]}}
errors = []
t0 = time.time()

for subj in sorted(os.listdir(f'{FIT3D_RAW}/train')):
    j3d_path = Path(FIT3D_RAW) / 'train' / subj / 'joints3d_25'
    if not j3d_path.exists():
        print(f'  ⚠️  {subj}: no joints3d_25/'); continue

    exercise_files = sorted(j3d_path.glob('*.json'))
    print(f'{subj} ({len(exercise_files)} exercises) ...', end=' ', flush=True)

    for jf in exercise_files:
        exercise = jf.stem
        try:
            joints = load_joints3d(jf)
            if joints is None:
                errors.append(f'{subj}/{exercise}: parse failed'); continue
            angles = compute_angles_sequence(joints, ANGLE_TRIPLETS)
            for ang_name, vals in angles.items():
                angle_stats[exercise][ang_name].extend(vals.tolist())
        except Exception as e:
            errors.append(f'{subj}/{exercise}: {e}')
    print('done')

print(f'\n✅ Done in {time.time()-t0:.0f}s  ({len(angle_stats)} exercises)')
if errors:
    print(f'   ⚠️  {len(errors)} errors:')
    for e in errors[:5]: print(f'      {e}')

In [ ]:
# ── Build ANGLE_RANGES and save ───────────────────────────────────────────────
ANGLE_RANGES = {}

for exercise, angle_dict in sorted(angle_stats.items()):
    ANGLE_RANGES[exercise] = {}
    for angle_name, values in sorted(angle_dict.items()):
        arr = np.array(values)
        ANGLE_RANGES[exercise][angle_name] = {
            'p5':       round(float(np.percentile(arr,  5)), 1),
            'p25':      round(float(np.percentile(arr, 25)), 1),
            'p50':      round(float(np.percentile(arr, 50)), 1),
            'p75':      round(float(np.percentile(arr, 75)), 1),
            'p95':      round(float(np.percentile(arr, 95)), 1),
            'min':      round(float(arr.min()), 1),
            'max':      round(float(arr.max()), 1),
            'n_frames': len(arr),
        }

out_path = f'{ANGLE_TEMPLATES_DIR}/golden_angle_ranges.json'
with open(out_path, 'w') as f:
    json.dump(ANGLE_RANGES, f, indent=2)

print(f'✅ Saved → {out_path}')
print(f'   {len(ANGLE_RANGES)} exercises, {len(ANGLE_TRIPLETS)} angles each')

# Preview
for ex in ['squat', 'pushup', 'dumbbell_biceps_curls']:
    if ex not in ANGLE_RANGES: continue
    print(f'\n  [{ex}]')
    for ang in ['left_knee_angle', 'left_elbow_angle']:
        if ang not in ANGLE_RANGES[ex]: continue
        s = ANGLE_RANGES[ex][ang]
        print(f'    {ang:30s}  p5={s["p5"]:5.1f}°  p50={s["p50"]:5.1f}°  p95={s["p95"]:5.1f}°')

---
## ✅ Step 12 — Validate Rep Counter with Fit3D Ground Truth

**Method:** Rep counter v5 — autocorrelation period-first angle selection.

| Technique | Purpose |
|---|---|
| Autocorrelation over all 12 angles | Find the angle with the clearest periodic signal |
| Adaptive Gaussian smoothing (σ = fps/20 for large ROM, fps/10 for small) | Remove noise without blurring rep peaks |
| Multi-distance search `[p//3, p//2, 2p//3, p]` | Handle harmonic detection in autocorrelation |
| Score = regularity + 0.5/(n+1) | Prefer more peaks when spacing is equally regular |
| Median vote across top-3 ROM angles | Robustness via multi-angle consensus |

**Metric:** `accuracy = max(0, 1 − |predicted − gt| / gt)`  (1.0 = perfect)

In [ ]:
import os, json, time
import numpy as np
from pathlib import Path
from collections import defaultdict
from scipy.signal import find_peaks
from scipy.ndimage import gaussian_filter1d
from numpy.fft import fft, ifft

# ── Load golden angle ranges (safe if 11c already ran in this session) ────────
if 'ANGLE_RANGES' not in dir() or not ANGLE_RANGES:
    with open(f'{ANGLE_TEMPLATES_DIR}/golden_angle_ranges.json') as f:
        ANGLE_RANGES = json.load(f)
    print(f'✅ ANGLE_RANGES loaded from file ({len(ANGLE_RANGES)} exercises)')
else:
    print(f'✅ ANGLE_RANGES already in memory ({len(ANGLE_RANGES)} exercises)')

# ── Ground-truth rep count ────────────────────────────────────────────────────
def get_gt_rep_count(entry):
    """Fit3D rep_ann.json format: list of peak frame indices per rep."""
    if isinstance(entry, list):  return len(entry)        # ← Fit3D native format
    if isinstance(entry, int):   return entry
    if isinstance(entry, dict):
        for k in ['reps', 'rep_count', 'count', 'n_reps', 'num_reps']:
            if k in entry: return int(entry[k])
        for k in ['intervals', 'segments', 'boundaries']:
            if k in entry: return len(entry[k])
    return None

# ── Spacing regularity (lower = more evenly spaced peaks) ────────────────────
def _peak_spacing_regularity(peak_indices):
    if len(peak_indices) < 2: return float('inf')
    gaps = np.diff(peak_indices)
    return float(np.std(gaps) / (np.mean(gaps) + 1e-8))

# ── Rep counter v5 ────────────────────────────────────────────────────────────
def count_reps_v5(angles_dict, exercise, angle_ranges, fps=50):
    """
    Autocorrelation period-first rep counter.

    1. Scan ALL 12 angles for the strongest autocorrelation period.
    2. Use that angle + period-derived distances as primary count.
    3. Median-vote with top-3 ROM angles as secondary.
    """
    if not angles_dict:
        return 0, []

    # ── 1. Find angle with strongest periodic autocorrelation ─────────────────
    best_ang, best_period, best_strength = None, None, 0.0

    for ang_name, seq in angles_dict.items():
        seq_arr = np.array(seq)
        if len(seq_arr) < 20: continue

        if exercise in angle_ranges and ang_name in angle_ranges[exercise]:
            rom   = angle_ranges[exercise][ang_name]['p95'] - angle_ranges[exercise][ang_name]['p5']
            sigma = fps / 20.0 if rom >= 60 else fps / 10.0
        else:
            sigma = fps / 10.0

        smoothed = gaussian_filter1d(seq_arr, sigma=sigma)
        n   = len(smoothed)
        sig = smoothed - smoothed.mean()
        std = sig.std()
        if std < 1e-6: continue
        sig /= std

        autocorr = np.real(ifft(np.abs(fft(sig, n=2*n))**2))[:n]
        autocorr /= (autocorr[0] + 1e-12)

        min_lag = max(1, int(fps * 0.5))
        max_lag = min(int(fps * 5.0), n // 2)
        if min_lag >= max_lag: continue

        peaks_ac, props_ac = find_peaks(autocorr[min_lag:max_lag], prominence=0.05)
        if len(peaks_ac) == 0: continue

        best_idx = peaks_ac[np.argmax(props_ac['prominences'])]
        period_f  = min_lag + best_idx
        strength  = autocorr[period_f]

        if strength > best_strength:
            best_strength, best_period, best_ang = strength, period_f, ang_name

    # ── 2. Count helper ───────────────────────────────────────────────────────
    def _count_angle(ang_name, period_hint):
        seq_arr = np.array(angles_dict[ang_name])
        if exercise in angle_ranges and ang_name in angle_ranges[exercise]:
            s          = angle_ranges[exercise][ang_name]
            rom        = s['p95'] - s['p5']
            iqr        = s['p75'] - s['p25']
            sigma      = fps / 20.0 if rom >= 60 else fps / 10.0
            prominence = max(5.0, iqr * 0.25)
        else:
            sigma, prominence = fps / 10.0, 10.0

        smoothed = gaussian_filter1d(seq_arr, sigma=sigma)

        if period_hint is not None:
            distances = sorted(set([
                max(8, period_hint // 3),
                max(8, period_hint // 2),
                max(8, period_hint * 2 // 3),
                max(8, period_hint),
            ]))
        else:
            distances = [int(fps * 0.3), int(fps * 0.4), int(fps * 0.5)]

        best_count, best_score = None, float('inf')
        for dist in distances:
            for direction in [1, -1]:
                peaks_f, _ = find_peaks(direction * smoothed, prominence=prominence, distance=dist)
                n_p = len(peaks_f)
                if n_p == 0: continue
                score = _peak_spacing_regularity(peaks_f) + 0.5 / (n_p + 1)
                if score < best_score:
                    best_score, best_count = score, n_p
        return best_count

    # ── 3. Primary: best-period angle ─────────────────────────────────────────
    counts = []
    if best_ang and best_strength >= 0.25:
        c = _count_angle(best_ang, best_period)
        if c: counts.append(c)

    # ── 4. Secondary: top-3 by ROM (excluding best_ang) ──────────────────────
    if exercise in angle_ranges:
        ranked = sorted(
            angle_ranges[exercise],
            key=lambda a: angle_ranges[exercise][a]['p95'] - angle_ranges[exercise][a]['p5'],
            reverse=True
        )
    else:
        ranked = list(angles_dict.keys())

    for ang in [a for a in ranked if a in angles_dict and a != best_ang][:3]:
        c = _count_angle(ang, None)
        if c: counts.append(c)

    if not counts:
        return 0, []
    return int(np.median(counts)), []

print('✅ All rep-counter functions defined (v5)')

In [ ]:
# ── Run validation across all train subjects ──────────────────────────────────
print('Running rep counter validation (v5) ...')
validation_results = []
t0 = time.time()

for subj in sorted(os.listdir(f'{FIT3D_RAW}/train')):
    subj_path = Path(FIT3D_RAW) / 'train' / subj
    rep_file  = subj_path / 'rep_ann.json'
    j3d_path  = subj_path / 'joints3d_25'
    if not rep_file.exists() or not j3d_path.exists(): continue

    with open(rep_file) as f:
        rep_ann = json.load(f)

    for exercise_json in sorted(j3d_path.glob('*.json')):
        exercise = exercise_json.stem
        gt_entry = rep_ann.get(exercise)
        if gt_entry is None: continue
        gt_reps = get_gt_rep_count(gt_entry)
        if gt_reps is None or gt_reps == 0: continue

        joints = load_joints3d(exercise_json)
        if joints is None: continue

        angles   = compute_angles_sequence(joints, ANGLE_TRIPLETS)
        pred, _  = count_reps_v5(angles, exercise, ANGLE_RANGES, fps=50)
        acc      = max(0.0, 1.0 - abs(pred - gt_reps) / gt_reps)

        validation_results.append({
            'subject':   subj,
            'exercise':  exercise,
            'gt_reps':   gt_reps,
            'pred_reps': pred,
            'accuracy':  round(acc, 3),
        })

print(f'Done in {time.time()-t0:.0f}s  ({len(validation_results)} sequences validated)')

# ── Summary ───────────────────────────────────────────────────────────────────
accs = [r['accuracy'] for r in validation_results]
print(f'\n=== Rep Counter Validation (v5) ===')
print(f'  Sequences:        {len(accs)}')
print(f'  Mean accuracy:    {np.mean(accs):.3f}  ({np.mean(accs)*100:.1f}%)')
print(f'  Perfect (100%):   {sum(1 for a in accs if a==1.0)/len(accs)*100:.1f}%')
print(f'  ≥ 90% accuracy:   {sum(1 for a in accs if a>=0.9)/len(accs)*100:.1f}%')
print(f'  ≥ 70% accuracy:   {sum(1 for a in accs if a>=0.7)/len(accs)*100:.1f}%')

per_ex = defaultdict(list)
for r in validation_results: per_ex[r['exercise']].append(r['accuracy'])

print('\n  Per-exercise (worst → best):')
for ex, ex_accs in sorted(per_ex.items(), key=lambda kv: np.mean(kv[1])):
    print(f'    {ex:40s}  {np.mean(ex_accs):.3f}  (n={len(ex_accs)})')

In [ ]:
# ── Save validation results ───────────────────────────────────────────────────
accs = [r['accuracy'] for r in validation_results]

out = {
    'summary': {
        'n_sequences':   len(validation_results),
        'mean_accuracy': round(float(np.mean(accs)), 4),
        'pct_perfect':   round(sum(1 for a in accs if a==1.0) / len(accs) * 100, 1),
        'pct_gte90':     round(sum(1 for a in accs if a>=0.9) / len(accs) * 100, 1),
        'pct_gte70':     round(sum(1 for a in accs if a>=0.7) / len(accs) * 100, 1),
        'method':        'v5_autocorr_period_first_adaptive_smooth_median_vote',
        'known_limits':  ['dumbbell_curl_trifecta', 'man_maker'],
    },
    'results': validation_results,
}

save_path = f'{EVAL_DIR}/rep_counter_validation.json'
with open(save_path, 'w') as f:
    json.dump(out, f, indent=2)

sz = os.path.getsize(save_path) / 1024
print(f'✅ Saved → {save_path}  ({sz:.0f} KB)')

# Confirm golden angle ranges also present
ranges_path = f'{ANGLE_TEMPLATES_DIR}/golden_angle_ranges.json'
sz2 = os.path.getsize(ranges_path) / 1024
print(f'✅ Golden templates → {ranges_path}  ({sz2:.0f} KB)')

---
## P02 Complete — Summary

| Deliverable | File | Used in |
|---|---|---|
| Golden angle templates | `datasets/fit3d/angle_templates/golden_angle_ranges.json` | P03 form scorer |
| Rep counter validation | `data/eval/rep_counter_validation.json` | Thesis Step 12 |

### Rep Counter Accuracy Progression
| Version | Key change | Mean Acc |
|---|---|---|
| v1 baseline | `find_peaks`, fixed params | 63.2% |
| v2 | Gaussian smooth + adaptive prominence | 65.4% |
| v3 | Multi-distance regularity voting | 67.6% |
| v4 | Autocorrelation period estimation | 71.6% |
| **v5** | **Period-first angle selection + harmonic-safe distances** | **~72%+** |

### Known limitations
- `dumbbell_curl_trifecta` and `man_maker`: one annotated rep = 3–4 chained sub-movements.
  Single-angle peak detection fundamentally overcounts these. Document as thesis limitation.

### What's next — P03
- Hook `golden_angle_ranges.json` into `form_scorer.py` as `ANGLE_RANGES`
- Run end-to-end evaluation on the Vicon TEST set (s02, s12, s13)